# 04 — Model Baseline (ResNet-18)

In [ ]:
# --- Setup ---
import sys, os
from pathlib import Path

# Colab: mount Drive and point to repo root
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = Path('/content/drive/MyDrive/deepfake-detection')
except ImportError:
    # Local: use DEEPFAKE_REPO_ROOT env var or cwd
    REPO_ROOT = Path(os.environ.get('DEEPFAKE_REPO_ROOT', str(Path.cwd())))

sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as T
import matplotlib.pyplot as plt

from configs.paths import (
    FRAMES_ROOT, TRAIN_CSV, VAL_CSV, TEST_CSV,
    MODEL_DIR, RESULTS_CSV, RESULTS_JSON_DIR
)

# Load split DataFrames
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

IMG_SIZE   = 224
NUM_FRAMES = 16

# ImageNet-normalisation transforms
train_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("REPO_ROOT :", REPO_ROOT)
print("train_df  :", len(train_df), "rows")
print("val_df    :", len(val_df),   "rows")
print("test_df   :", len(test_df),  "rows")


In [ ]:
import time
from datetime import datetime

# --- SMOKE_TEST mode ---
# True  = 50 train / 20 val / 20 test, 1 + 1 epochs -- fast end-to-end validation
# False = full dataset, production training schedule
SMOKE_TEST = False

# --- src + config imports ---
from configs.experiments import RESNET18_CONFIG
from src.datasets        import DeepfakeBinaryDataset, get_dataloaders
from src.models          import ResNetBinaryVideoClassifier
from src.training        import train_two_stage, pick_device
from src.evaluation      import evaluate, per_manipulation_breakdown
from src.logging         import make_run_id, append_run_to_csv, write_run_json
from src.visualization   import plot_training_history, plot_confusion_matrix

RUN_ID = make_run_id("resnet18")
CONFIG = dict(RESNET18_CONFIG)  # copy so local tweaks don't mutate module state
CONFIG["run_id"]          = RUN_ID
CONFIG["checkpoint_dir"]  = str(MODEL_DIR / "checkpoints" / RUN_ID)

device = pick_device()
print(f"Run ID : {RUN_ID}")
print(f"Device : {device}")
print(f"Smoke  : {SMOKE_TEST}")


### Model + training pipeline

In [ ]:
# --- Dataset subsampling ---
# In SMOKE_TEST mode: 50/20/20; otherwise use full splits
if SMOKE_TEST:
    train_df_model = train_df.sample(n=min(50, len(train_df)), random_state=42).reset_index(drop=True)
    val_df_model   = val_df.sample(n=min(20, len(val_df)),     random_state=42).reset_index(drop=True)
    test_df_model  = test_df.sample(n=min(20, len(test_df)),   random_state=42).reset_index(drop=True)
    print("SMOKE subset sizes:", len(train_df_model), len(val_df_model), len(test_df_model))
else:
    train_df_model = train_df.copy().reset_index(drop=True)
    val_df_model   = val_df.copy().reset_index(drop=True)
    test_df_model  = test_df.copy().reset_index(drop=True)
    print("Train samples:", len(train_df_model))
    print("Val samples  :", len(val_df_model))
    print("Test samples :", len(test_df_model))


In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(
    train_df=train_df_model,
    val_df=val_df_model,
    test_df=test_df_model,
    frames_root=FRAMES_ROOT,
    train_transform=train_transform,
    eval_transform=val_transform,
    batch_size=CONFIG["batch_size"],
    num_frames=CONFIG["num_frames"],
    num_workers=CONFIG["num_workers"] if not SMOKE_TEST else 0,
    weighted_sampler=CONFIG["weighted_sampler"],
)
print("Train loader batches:", len(train_loader))
print("Val loader batches  :", len(val_loader))
print("Test loader batches :", len(test_loader))


In [ ]:
# --- Model instantiation ---
model = ResNetBinaryVideoClassifier(dropout=0.3)

# class-weighted loss, computed from training split
train_counts = train_df_model["binary_target"].value_counts().sort_index().to_dict()
num_real = train_counts.get(0, 0)
num_fake = train_counts.get(1, 0)
total_n  = num_real + num_fake
class_weights = torch.tensor(
    [total_n / (2 * max(num_real, 1)), total_n / (2 * max(num_fake, 1))],
    dtype=torch.float32,
)
CONFIG["class_weights"] = class_weights
print("Class weights:", class_weights.tolist())

# Smoke-test schedule override
if SMOKE_TEST:
    CONFIG["epochs_stage1"] = 1
    CONFIG["epochs_stage2"] = 1


In [ ]:
batch_frames, batch_labels = next(iter(train_loader))

batch_frames = batch_frames.to(device)
batch_labels = batch_labels.to(device)

with torch.no_grad():
    outputs = model(batch_frames)

print("Input shape :", batch_frames.shape)
print("Output shape:", outputs.shape)
print("Labels shape:", batch_labels.shape)

In [ ]:
t0 = time.time()
best_state, history = train_two_stage(model, train_loader, val_loader, CONFIG, device=device)
train_minutes = (time.time() - t0) / 60.0
model.load_state_dict(best_state)
print(f"Training time: {train_minutes:.1f} min")


In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
ffpp_test = evaluate(model, test_loader, criterion, device)

print(f"FF++ Test  accuracy : {ffpp_test['accuracy']:.4f}")
print(f"FF++ Test  F1       : {ffpp_test['f1']:.4f}")
print(f"FF++ Test  AUC      : {ffpp_test['auc']:.4f}")

from sklearn.metrics import classification_report
print(classification_report(ffpp_test["y_true"], ffpp_test["y_pred"], target_names=["real", "fake"]))

# Per-manipulation breakdown
FAKE_CLASSES = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures", "FaceShifter"]
per_manip = per_manipulation_breakdown(
    test_df_model, ffpp_test["y_pred"], ffpp_test["y_probs"], FAKE_CLASSES
)
for manip, vals in per_manip.items():
    auc = vals.get("auc")
    if auc is not None:
        print(f"  {manip:16s} AUC={auc:.4f}  F1={vals['f1']:.4f}  n={vals['n_videos']}")
    else:
        print(f"  {manip:16s} AUC=--      F1={vals['f1']:.4f}  n={vals['n_videos']}")


In [ ]:
fig = plot_confusion_matrix(ffpp_test["y_true"], ffpp_test["y_pred"], labels=["real", "fake"])
fig.show()
fig = plot_training_history(history)
fig.show()


In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "resnet18_binary_deepfake_detector_best.pth"
torch.save(model.state_dict(), model_path)

print("Model saved to:", model_path)

In [ ]:
env_kind = "local-m5" if device.type == "mps" else ("colab-pro" if device.type == "cuda" else "cpu")

metrics_row = {
    "environment":            env_kind,
    "device":                 str(device.type),
    "ffpp_val_acc":           history["stage2"]["val_acc"][-1] if history["stage2"]["val_acc"] else None,
    "ffpp_val_f1":            None,
    "ffpp_test_acc":          ffpp_test["accuracy"],
    "ffpp_test_precision":    ffpp_test["precision"],
    "ffpp_test_recall":       ffpp_test["recall"],
    "ffpp_test_f1":           ffpp_test["f1"],
    "ffpp_test_auc":          ffpp_test["auc"],
    "celebdf_acc":            None,
    "celebdf_f1":             None,
    "celebdf_auc":            None,
    "generalization_gap_auc": None,
    "train_time_minutes":     round(train_minutes, 2),
    "notes":                  "smoke" if SMOKE_TEST else "",
}

append_run_to_csv(RUN_ID, CONFIG, metrics_row, RESULTS_CSV)

json_payload = {
    "run_id":             RUN_ID,
    "timestamp":          datetime.now().isoformat(),
    "environment":        {"kind": env_kind, "device": str(device)},
    "config":             {k: v for k, v in CONFIG.items() if k != "class_weights"},
    "training_history":   history,
    "ffpp_test":          {k: ffpp_test[k] for k in ("accuracy", "precision", "recall", "f1", "auc", "confusion_matrix")},
    "per_manipulation":   per_manip,
    "celebdf_test":       None,
    "generalization_gap": None,
    "checkpoints":        {"best": str(MODEL_DIR / "checkpoints" / RUN_ID / f"{RUN_ID}_best.pth")},
    "metrics":            metrics_row,
    "notes":              "smoke" if SMOKE_TEST else "",
}
write_run_json(RUN_ID, json_payload, RESULTS_JSON_DIR)
print("Logged run:", RUN_ID)
print("CSV:", RESULTS_CSV)
